# Part A: mZAR Remittance Prototype

A worker in South Africa sends value to a family member in Zimbabwe.

**Payment journey:** deposit 1,000 local units → transfer 750 mZAR → redeem 750 local units.

All money and payouts are simulated. The prototype uses one rand-denominated unit.

**Reserve rule:** 1 mZAR is backed by 1 simulated rand. The Reserve agent signs every mint and every payout receipt.

In [1]:
%pip install flask ecdsa

Note: you may need to restart the kernel to use updated packages.


## 1. Setup

ECDSA authorises instructions. SHA-256 identifies transactions and blocks.

In [2]:
import json
import hashlib
import time
import secrets
import copy
import threading

from html import escape
from urllib.request import urlopen
from ecdsa import SigningKey, VerifyingKey, SECP256k1
from ecdsa.keys import BadSignatureError
from flask import Flask, request, jsonify, render_template_string
from IPython.display import display, HTML

ASSET = "mZAR"
DIFFICULTY = 2
REQUIRED_CONFIRMATIONS = 2
GENESIS_PREV = "0" * 64
HEX_DIGITS = set("0123456789abcdef")

def canonical(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False
    ).encode("utf-8")


def digest(value):
    return hashlib.sha256(canonical(value)).hexdigest()


def sha256_hex(data):
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def positive_amount(value):
    if type(value) is not int or value <= 0:
        raise ValueError("Amount must be a positive integer")


def valid_address(value):
    """An address is exactly 40 lowercase hexadecimal characters."""
    return (
        isinstance(value, str)
        and len(value) == 40
        and set(value) <= HEX_DIGITS
    )


def require_address(value, label="wallet address"):
    if not valid_address(value):
        raise ValueError(
            f"Invalid {label}: use 40 lowercase hexadecimal characters"
        )


print("Setup ready")
print("Asset:", ASSET)
print("Mining difficulty:", DIFFICULTY)
print("Required payout confirmations:", REQUIRED_CONFIRMATIONS)

Setup ready
Asset: mZAR
Mining difficulty: 2
Required payout confirmations: 2


In [3]:
print("GENESIS_PREV:", GENESIS_PREV)

GENESIS_PREV: 0000000000000000000000000000000000000000000000000000000000000000


## 2. Wallets and authorisation

Three wallets are created:

In [4]:
class Wallet:
    def __init__(self, signing_key=None, label="Wallet"):
        if signing_key is None:
            signing_key = SigningKey.generate(
                curve=SECP256k1
            )

        self.label = label
        self._signing_key = signing_key
        self._verifying_key = (
            signing_key.get_verifying_key()
        )

    @classmethod
    def create(cls, label="Wallet"):
        signing_key = SigningKey.generate(
            curve=SECP256k1
        )
        return cls(signing_key, label)

    @property
    def address(self):
        public_key_bytes = (
            self._verifying_key.to_string()
        )

        return hashlib.sha256(
            public_key_bytes
        ).hexdigest()[:40]

    @property
    def public_key_hex(self):
        return self._verifying_key.to_string().hex()

    def sign(self, message):
        signature = self._signing_key.sign_deterministic(
            message,
            hashfunc=hashlib.sha256
        )

        return signature.hex()

    def sign_record(self, body):
        """Sign a record (mint or receipt) as canonical JSON."""
        return self.sign(canonical(body))

    @staticmethod
    def verify(message, signature_hex, public_key_hex):
        try:
            verifying_key = VerifyingKey.from_string(
                bytes.fromhex(public_key_hex),
                curve=SECP256k1
            )

            return verifying_key.verify(
                bytes.fromhex(signature_hex),
                message,
                hashfunc=hashlib.sha256
            )

        except (BadSignatureError, ValueError, TypeError):
            return False

    def instruction(self, kind, recipient, amount, nonce, timestamp=None):
        positive_amount(amount)

        body = {
            "kind": kind,
            "asset": ASSET,
            "sender": self.address,
            "recipient": recipient,
            "amount": amount,
            "timestamp": (
                int(time.time())
                if timestamp is None
                else timestamp
            ),
            "nonce": nonce,
            "public_key": self.public_key_hex
        }

        return {
            **body,
            "signature": self.sign(canonical(body)),
            "tx_id": digest(body)
        }


def signed_body(tx):
    """Fields covered by the signature: everything except signature and tx_id."""
    fields = (
        "kind",
        "asset",
        "sender",
        "recipient",
        "amount",
        "timestamp",
        "nonce",
        "public_key"
    )

    return {key: tx[key] for key in fields}


def verify_instruction(tx):
    try:
        body = signed_body(tx)

        positive_amount(body["amount"])

        if body["asset"] != ASSET:
            return False

        if body["kind"] not in ("transfer", "burn"):
            return False

        if (
            type(body["nonce"]) is not int
            or body["nonce"] < 0
        ):
            return False

        if (
            type(body["timestamp"]) not in (int, float)
            or body["timestamp"] <= 0
        ):
            return False

        expected_address = hashlib.sha256(
            bytes.fromhex(body["public_key"])
        ).hexdigest()[:40]

        return (
            expected_address == body["sender"]
            and tx["tx_id"] == digest(body)
            and Wallet.verify(
                canonical(body),
                tx["signature"],
                body["public_key"]
            )
        )

    except (
        KeyError,
        ValueError,
        TypeError,
        BadSignatureError
    ):
        return False


def verify_record(record, issuer_public_key):
    """Verify an issuer-signed record: a mint or a payout receipt."""
    body = {
        key: value
        for key, value in record.items()
        if key not in ("signature", "tx_id", "receipt_id")
    }

    record_id = record.get("tx_id", record.get("receipt_id"))

    return (
        record_id == digest(body)
        and Wallet.verify(
            canonical(body),
            record.get("signature", ""),
            issuer_public_key
        )
    )


sender = Wallet.create("Sender")
recipient = Wallet.create("Recipient")
reserve_agent = Wallet.create("Reserve agent (issuer)")

print("Sender address:       ", sender.address)
print("Recipient address:    ", recipient.address)
print("Reserve agent address:", reserve_agent.address)

Sender address:        88b0f7e17f7b57b18f4c67eace5e807059200d9e
Recipient address:     8088396b69b74481b8575fda8a4bd6a176b95f49
Reserve agent address: 954462896f01b2fc449b56fa30968f837d2bb881


## 3. Blockchain and mempool

In [5]:
class Block:
    def __init__(
        self,
        index,
        transactions,
        previous_hash,
        nonce=0,
        timestamp=None,
        hash=None
    ):
        self.index = index
        self.timestamp = (
            time.time()
            if timestamp is None
            else timestamp
        )
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = (
            self.compute_hash()
            if hash is None
            else hash
        )

    def payload_for_hash(self):
        """Fields committed to by the hash; the stored hash is not included."""
        return {
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }

    def compute_hash(self):
        block_string = json.dumps(
            self.payload_for_hash(),
            sort_keys=True,
            separators=(",", ":")
        )

        return sha256_hex(block_string)

    def to_dict(self):
        return copy.deepcopy({
            **self.payload_for_hash(),
            "hash": self.hash
        })

    @classmethod
    def from_dict(cls, data):
        return cls(
            index=data["index"],
            transactions=data["transactions"],
            previous_hash=data["previous_hash"],
            nonce=data["nonce"],
            timestamp=data["timestamp"],
            hash=data["hash"]
        )


def create_genesis_block():
    return Block(
        index=0,
        transactions=[],
        previous_hash=GENESIS_PREV,
        nonce=0
    )


def create_linked_block(previous, transactions):
    return Block(
        index=previous.index + 1,
        transactions=copy.deepcopy(list(transactions)),
        previous_hash=previous.hash
    )


def check_genesis_previous_hash(block):
    """Lecture 05: the genesis block must point to exactly 64 zeros."""
    return (
        block.index == 0
        and block.previous_hash == GENESIS_PREV
        and len(block.previous_hash) == 64
        and set(block.previous_hash) == {"0"}
    )


def is_hash_valid(block, difficulty=None):
    if block.hash != block.compute_hash():
        return False

    if difficulty is None:
        return True

    return block.hash.startswith("0" * difficulty)


class Blockchain:
    def __init__(self):
        self.chain = [create_genesis_block()]

    def tip(self):
        return self.chain[-1]

    def append_block(self, block, difficulty=None):
        tip = self.tip()

        if block.index != tip.index + 1:
            raise ValueError("Bad block index")

        if block.previous_hash != tip.hash:
            raise ValueError("Bad previous hash")

        if block.hash != block.compute_hash():
            raise ValueError("Bad block hash")

        if difficulty is not None:
            if not block.hash.startswith(
                "0" * difficulty
            ):
                raise ValueError("Difficulty not met")

        self.chain.append(block)

    def verify_chain(self, difficulty=None):
        if not self.chain:
            return False

        genesis = self.chain[0]

        if not check_genesis_previous_hash(genesis):
            return False

        if genesis.hash != genesis.compute_hash():
            return False

        for index in range(1, len(self.chain)):
            current = self.chain[index]
            previous = self.chain[index - 1]

            if current.index != index:
                return False

            if current.hash != current.compute_hash():
                return False

            if current.previous_hash != previous.hash:
                return False

            if difficulty is not None:
                if not current.hash.startswith(
                    "0" * difficulty
                ):
                    return False

        return True


def mine_block(block, difficulty):
    if difficulty < 0:
        raise ValueError(
            "Difficulty must be non-negative"
        )

    prefix = "0" * difficulty
    nonce = 0
    start = time.perf_counter()

    while True:
        block.nonce = nonce
        candidate_hash = block.compute_hash()

        if candidate_hash.startswith(prefix):
            block.hash = candidate_hash
            elapsed = time.perf_counter() - start

            return block, nonce + 1, elapsed

        nonce += 1


# Adapted from Assignment 3.
# Signed dictionaries replace the earlier Transaction objects.

class Mempool:
    def __init__(self):
        self._transactions = []

    def add(self, tx):
        if (
            not verify_instruction(tx)
            or tx["kind"] != "transfer"
        ):
            return False

        self._transactions.append(
            copy.deepcopy(tx)
        )

        return True

    def contains(self, tx_id):
        return any(
            tx["tx_id"] == tx_id
            for tx in self._transactions
        )

    def get_all(self):
        return copy.deepcopy(self._transactions)

    def __len__(self):
        return len(self._transactions)

    def __iter__(self):
        return iter(self.get_all())

    def clear(self):
        self._transactions.clear()


def block_to_dict(block):
    return block.to_dict()


def print_block(block, title):
    """Print every field of a block, one per line."""
    print(f"=== {title} ===")
    print("  index:", block.index)
    print("  timestamp:", block.timestamp)
    print("  transactions:", block.transactions)
    print("  previous_hash:", block.previous_hash)
    print("  nonce:", block.nonce)
    print("  hash:", block.hash)


print("Blockchain and mempool components ready\n")

# Genesis block: fixed anchor of the chain.
genesis = create_genesis_block()

print_block(genesis, "Genesis")
print(
    "  previous_hash is 64 zeros:",
    check_genesis_previous_hash(genesis)
)
print()

# Block 1 (worked example, demo only): linked to the genesis block and mined
# with proof of work. It is not added to the payment chain.
demo_block = create_linked_block(
    genesis,
    [{"sender": "Sender", "recipient": "Recipient", "amount": 10}]
)
demo_block, demo_attempts, demo_elapsed = mine_block(
    demo_block,
    DIFFICULTY
)

print_block(demo_block, "Block 1)")
print(
    "  links to genesis (previous_hash == genesis hash):",
    demo_block.previous_hash == genesis.hash
)
print(
    f"  proof of work met (difficulty {DIFFICULTY}):",
    is_hash_valid(demo_block, DIFFICULTY),
    f"after {demo_attempts} attempts"
)
print()

# A genesis block that points anywhere else must fail the check.
tampered_genesis = create_genesis_block()
tampered_genesis.previous_hash = "0" * 63 + "1"

assert check_genesis_previous_hash(genesis)
assert not check_genesis_previous_hash(tampered_genesis)
assert demo_block.previous_hash == genesis.hash
assert is_hash_valid(demo_block, DIFFICULTY)

print(
    "Tampered genesis (previous_hash ends in 1) passes check:",
    check_genesis_previous_hash(tampered_genesis)
)

Blockchain and mempool components ready

=== Genesis ===
  index: 0
  timestamp: 1791733620.5359356
  transactions: []
  previous_hash: 0000000000000000000000000000000000000000000000000000000000000000
  nonce: 0
  hash: debe2f70428dc5a7a7f56c51ec32393d4944ba9d24e922f5d463998913996381
  previous_hash is 64 zeros: True

=== Block 1) ===
  index: 1
  timestamp: 1791733620.5385633
  transactions: [{'sender': 'Sender', 'recipient': 'Recipient', 'amount': 10}]
  previous_hash: debe2f70428dc5a7a7f56c51ec32393d4944ba9d24e922f5d463998913996381
  nonce: 70
  hash: 00caef94a0da5bf3edf5e8fa4637507db481af195c3b18fd5936c517d15e482f
  links to genesis (previous_hash == genesis hash): True
  proof of work met (difficulty 2): True after 71 attempts

Tampered genesis (previous_hash ends in 1) passes check: False


## 4. Reserve ledger and payment controls

Deposits issue tokens. Transfers move balances. Redemptions burn tokens.

In [6]:
class DuplicateTransaction(ValueError):
    """The same instruction was submitted twice (HTTP 409)."""


class RemittancePlatform:
    def __init__(self, issuer=None):
        # The Reserve agent holds the vault and signs mints and receipts.
        self.issuer = issuer or Wallet.create("Reserve agent (issuer)")

        self.blockchain = Blockchain()
        self.pool = Mempool()

        self.events = []
        self.receipts = {}
        self.last_mining = None

        self.ledger = self.empty_ledger()
        self.lock = threading.RLock()

    @staticmethod
    def empty_ledger():
        return {
            "vault_fiat": 0,
            "circulating": 0,
            "balances": {},
            "nonces": {}
        }

    @staticmethod
    def invariant(state):
        return (
            state["vault_fiat"]
            == state["circulating"]
            == sum(state["balances"].values())
            and state["vault_fiat"] >= 0
            and all(
                value >= 0
                for value in state["balances"].values()
            )
        )

    def invariant_ok(self):
        """Lecture 10: vault_fiat == circulating (and balances add up)."""
        return self.invariant(self.ledger)

    @property
    def chain(self):
        return [
            block.to_dict()
            for block in self.blockchain.chain
        ]

    @staticmethod
    def block_hash(block):
        return Block.from_dict(block).compute_hash()

    def _mine(self, transactions):
        block = create_linked_block(
            self.blockchain.tip(),
            transactions
        )

        mined, attempts, elapsed = mine_block(
            block,
            DIFFICULTY
        )

        self.blockchain.append_block(
            mined,
            difficulty=DIFFICULTY
        )

        # Evidence for the report: proof-of-work effort per block.
        self.last_mining = {
            "difficulty": DIFFICULTY,
            "nonce": mined.nonce,
            "attempts": attempts,
            "seconds": round(elapsed, 4)
        }

        return mined.to_dict()

    def apply(self, state, tx):
        amount = tx["amount"]
        positive_amount(amount)

        balances = state["balances"]
        nonces = state["nonces"]

        if tx["kind"] == "mint":
            if (
                tx["asset"] != ASSET
                or not verify_record(
                    tx,
                    self.issuer.public_key_hex
                )
            ):
                raise ValueError(
                    "Invalid mint record: missing or bad Reserve agent signature"
                )

            destination = tx["recipient"]
            require_address(destination, "mint recipient")

            state["vault_fiat"] += amount
            state["circulating"] += amount

            balances[destination] = (
                balances.get(destination, 0)
                + amount
            )

        else:
            if not verify_instruction(tx):
                raise ValueError(
                    "Invalid signature or transaction ID"
                )

            address = tx["sender"]

            if tx["nonce"] != nonces.get(address, 0):
                raise ValueError(
                    "Invalid nonce or repeated instruction"
                )

            if balances.get(address, 0) < amount:
                raise ValueError("Insufficient balance")

            if tx["kind"] == "transfer":
                destination = tx["recipient"]

                require_address(
                    destination,
                    "recipient address"
                )

                balances[address] -= amount

                balances[destination] = (
                    balances.get(destination, 0)
                    + amount
                )

            else:
                if tx["recipient"] != "FIAT_VAULT":
                    raise ValueError(
                        "Invalid redemption destination"
                    )

                balances[address] -= amount
                state["vault_fiat"] -= amount
                state["circulating"] -= amount

            nonces[address] = (
                nonces.get(address, 0) + 1
            )

        if not RemittancePlatform.invariant(state):
            raise ValueError(
                "Reserve or balance invariant failed"
            )

    def commit(self, transactions):
        candidate = copy.deepcopy(self.ledger)

        for tx in transactions:
            self.apply(candidate, tx)

        block = self._mine(transactions)
        self.ledger = candidate

        return block

    def onramp(self, address, amount):
        """Simulated deposit: vault_fiat and circulating rise 1:1.
        In production this sits behind KYC/AML checks and operator authentication."""
        positive_amount(amount)
        require_address(address)

        if self.pool:
            raise ValueError(
                "Mine pending transfers before changing reserves"
            )

        body = {
            "kind": "mint",
            "asset": ASSET,
            "recipient": address,
            "amount": amount,
            "timestamp": time.time(),
            "event_id": secrets.token_hex(16),
            "issuer": self.issuer.address
        }

        # The Reserve agent authorises the mint.
        tx = {
            **body,
            "signature": self.issuer.sign_record(body),
            "tx_id": digest(body)
        }

        block = self.commit([tx])

        event = {
            "event": "onramp",
            "tx_id": tx["tx_id"],
            "block": block["index"],
            "amount": amount,
            "deposited_local": amount,
            "minted_mzar": amount,
            "vault_fiat_after": self.ledger["vault_fiat"],
            "circulating_after": self.ledger["circulating"],
            "issuer_signature": tx["signature"]
        }

        self.events.append(event)
        return event

    def submit(self, tx):
        if tx.get("kind") != "transfer":
            raise ValueError(
                "Only transfer instructions enter the mempool"
            )

        # Idempotency (Lecture 08): the same instruction cannot be queued twice.
        if (
            self.pool.contains(tx.get("tx_id"))
            or self.lookup(tx.get("tx_id")) is not None
        ):
            raise DuplicateTransaction(
                "Duplicate transaction: already pending or on chain"
            )

        projected = copy.deepcopy(self.ledger)

        for pending in self.pool:
            self.apply(projected, pending)

        self.apply(projected, tx)

        if not self.pool.add(tx):
            raise ValueError(
                "Invalid mempool instruction"
            )

        return {
            "message": "Transaction queued",
            "block_index": self.blockchain.tip().index + 1,
            "tx_id": tx["tx_id"],
            "status": "pending",
            "pending": len(self.pool)
        }

    def mine_pending(self):
        block = self.commit(self.pool.get_all())
        self.pool.clear()

        return {
            "message": "New block forged",
            "index": block["index"],
            "hash": block["hash"],
            "transactions": block["transactions"],
            "mining": self.last_mining
        }

    def lookup(self, tx_id):
        chain = self.chain

        for block in chain:
            for tx in block["transactions"]:
                if tx["tx_id"] == tx_id:
                    confirmations = (
                        len(chain) - block["index"]
                    )

                    return {
                        "tx_id": tx_id,
                        "status": (
                            "confirmed"
                            if confirmations >= REQUIRED_CONFIRMATIONS
                            else "mined"
                        ),
                        "block": block["index"],
                        "block_hash": block["hash"],
                        "confirmations": confirmations,
                        "required_confirmations": REQUIRED_CONFIRMATIONS,
                        "transaction": copy.deepcopy(tx)
                    }

        for tx in self.pool:
            if tx["tx_id"] == tx_id:
                return {
                    "tx_id": tx_id,
                    "status": "pending",
                    "block": None,
                    "confirmations": 0,
                    "required_confirmations": REQUIRED_CONFIRMATIONS,
                    "transaction": tx
                }

        return None

    def eligible_balance(self, address):
        # Only this wallet's own pending outgoing transfers block redemption.
        if any(tx["sender"] == address for tx in self.pool):
            raise ValueError(
                "This wallet has a pending outgoing transfer; mine it before redeeming"
            )

        chain = self.chain
        confirmed = self.empty_ledger()

        for block in chain:
            confirmations = (
                len(chain) - block["index"]
            )

            if confirmations >= REQUIRED_CONFIRMATIONS:
                for tx in block["transactions"]:
                    self.apply(confirmed, tx)

        if (
            confirmed["nonces"].get(address, 0)
            != self.ledger["nonces"].get(address, 0)
        ):
            raise ValueError(
                "Outgoing activity has not reached required confirmations"
            )

        return min(
            confirmed["balances"].get(address, 0),
            self.ledger["balances"].get(address, 0)
        )

    def offramp(self, tx):
        """Signed redemption: burn mZAR, reduce vault_fiat, issue a signed receipt."""
        if (
            tx.get("kind") != "burn"
            or not verify_instruction(tx)
        ):
            raise ValueError(
                "Invalid redemption authorisation"
            )

        positive_amount(tx["amount"])

        if (
            self.eligible_balance(tx["sender"])
            < tx["amount"]
        ):
            raise ValueError(
                "Funds have not reached required confirmations"
            )

        block = self.commit([tx])
        address = tx["sender"]

        # Link the payout back to the on-chain transfers that funded it.
        funding = [
            item["tx_id"]
            for chain_block in self.chain
            for item in chain_block["transactions"]
            if item["kind"] == "transfer"
            and item["recipient"] == address
        ]

        body = {
            "burn_tx_id": tx["tx_id"],
            "address": address,
            "burned_mzar": tx["amount"],
            "paid_local": tx["amount"],
            "currency": "ZAR demo units",
            "payout": "SIMULATED",
            "block": block["index"],
            "block_hash": block["hash"],
            "funding_tx_ids": funding,
            "vault_fiat_after": self.ledger["vault_fiat"],
            "circulating_after": self.ledger["circulating"],
            "timestamp": time.time(),
            "issuer": self.issuer.address
        }

        # The Reserve agent signs the payout receipt.
        receipt = {
            **body,
            "signature": self.issuer.sign_record(body),
            "receipt_id": digest(body)
        }

        self.receipts[receipt["receipt_id"]] = receipt
        return receipt

    def verify_chain(self, chain=None):
        """Check every block and replay the full ledger (hashes, links, PoW,
        issuer and wallet signatures, nonces, balances and the 1:1 rule)."""
        candidate = (
            self.chain
            if chain is None
            else chain
        )

        try:
            if not candidate:
                return False

            if chain is None:
                if not self.blockchain.verify_chain(
                    DIFFICULTY
                ):
                    return False

            state = self.empty_ledger()
            seen = set()

            for index, block in enumerate(candidate):
                previous = (
                    candidate[index - 1]["hash"]
                    if index
                    else GENESIS_PREV
                )

                if (
                    block["index"] != index
                    or block["previous_hash"] != previous
                    or self.block_hash(block)
                    != block["hash"]
                ):
                    return False

                # Genesis is a fixed anchor.
                if index > 0:
                    if not block["hash"].startswith(
                        "0" * DIFFICULTY
                    ):
                        return False

                for tx in block["transactions"]:
                    if tx["tx_id"] in seen:
                        return False

                    seen.add(tx["tx_id"])
                    self.apply(state, tx)

            return (
                self.invariant(state)
                and state == self.ledger
            )

        except (KeyError, ValueError, TypeError):
            return False

    # Short name kept so earlier code still works.
    verify = verify_chain

    def status(self):
        return {
            **copy.deepcopy(self.ledger),
            "invariant_ok": self.invariant_ok(),
            "chain_ok": self.verify_chain(),
            "blocks": len(self.blockchain.chain),
            "pending": len(self.pool)
        }


platform = RemittancePlatform(reserve_agent)

print("Genesis ready")
print("Reserve agent (issuer):", platform.issuer.address)
print(
    "Blockchain verified:",
    platform.blockchain.verify_chain(DIFFICULTY)
)
print("verify_chain():", platform.verify_chain())

Genesis ready
Reserve agent (issuer): 954462896f01b2fc449b56fa30968f837d2bb881
Blockchain verified: True
verify_chain(): True


## 5. Payment API and explorer

In [7]:
EXPLORER_HTML = """
<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>mZAR Remittance Explorer</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 25px;
            background: #f3f6fa;
            color: #183048;
        }
        article {
            background: white;
            padding: 18px;
            margin-bottom: 18px;
            border-left: 5px solid #167d9a;
            border-radius: 6px;
        }
        code {
            overflow-wrap: anywhere;
            word-break: break-all;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            table-layout: fixed;
        }
        th, td {
            padding: 8px;
            text-align: left;
            border-bottom: 1px solid #dde4eb;
            overflow-wrap: anywhere;
        }
    </style>
</head>
<body>
    <h1>mZAR Remittance Explorer</h1>
    <p>
        Chain length: {{ length }}
        | Pending transactions: {{ pending }}
        | verify_chain(): {{ verified }}
    </p>
    <p>
        Vault: R{{ vault_fiat }}
        | Circulating: {{ circulating }} mZAR
        | 1:1 invariant OK: {{ invariant_ok }}
    </p>
    <p>All amounts are simulated mZAR. Recent blocks appear first.</p>

    {% for block in blocks %}
    <article>
        <h2>Block {{ block.index }}</h2>
        <p><strong>Hash:</strong> <code>{{ block.hash }}</code></p>
        <p><strong>Previous hash:</strong>
           <code>{{ block.previous_hash }}</code></p>
        <p>
            Timestamp: {{ block.timestamp }}
            | Nonce: {{ block.nonce }}
            | Confirmations: {{ block.confirmations }}
        </p>

        <table>
            <tr>
                <th>Transaction ID</th>
                <th>From</th>
                <th>To</th>
                <th>Amount (mZAR)</th>
                <th>Event</th>
            </tr>

            {% for tx in block.rows %}
            <tr>
                <td><code>{{ tx.tx_id }}</code></td>
                <td>{{ tx.sender }}</td>
                <td>{{ tx.recipient }}</td>
                <td>{{ tx.amount }}</td>
                <td>{{ tx.kind }}</td>
            </tr>
            {% else %}
            <tr>
                <td colspan="5">
                    No payment events: genesis or confirmation block.
                </td>
            </tr>
            {% endfor %}
        </table>
    </article>
    {% endfor %}
</body>
</html>
"""


def explorer_rows(chain, limit=10):
    """Lecture 08 explorer rows: latest blocks first."""
    return [
        {
            "index": block["index"],
            "hash": block["hash"][:16] + "...",
            "n_tx": len(block.get("transactions", [])),
            "timestamp": block.get("timestamp")
        }
        for block in reversed(chain[-limit:])
    ]


def explorer_blocks(service, labels=None):
    """Blocks prepared for display, with readable party names."""
    names = {
        **(labels or {}),
        service.issuer.address: "Reserve agent",
        "FIAT_VAULT": "Reserve vault (payout)"
    }

    def party(address):
        name = names.get(address)

        if name and valid_address(address):
            return f"{name} ({address[:8]}…)"

        return name or address

    chain = service.chain
    blocks = []

    for block in reversed(chain):
        rows = []

        for tx in block["transactions"]:
            from_address = (
                service.issuer.address
                if tx["kind"] == "mint"
                else tx["sender"]
            )

            rows.append({
                "tx_id": tx["tx_id"],
                "sender": party(from_address),
                "recipient": party(tx["recipient"]),
                "amount": tx["amount"],
                "kind": tx["kind"]
            })

        blocks.append({
            **block,
            "confirmations": len(chain) - block["index"],
            "rows": rows
        })

    return blocks


def create_app(service, labels=None):
    application = Flask("mzar_capstone")

    def json_body():
        body = request.get_json(silent=True)

        if not isinstance(body, dict):
            raise ValueError(
                "Provide a JSON object"
            )

        return body

    # The local deposit endpoint represents a trusted simulated issuer.
    # In production it would sit behind KYC/AML checks and operator auth.
    @application.post("/onramp")
    def onramp_route():
        body = json_body()

        return jsonify(
            service.onramp(
                body.get("address"),
                body.get("amount")
            )
        ), 201

    # A8 endpoint, with the earlier /transfer name kept as an alias.
    @application.post("/transactions/new")
    @application.post("/transfer")
    def new_transaction():
        return jsonify(
            service.submit(json_body())
        ), 201

    @application.post("/mine")
    def mine():
        return jsonify(
            service.mine_pending()
        ), 200

    @application.post("/offramp")
    def offramp_route():
        return jsonify(
            service.offramp(json_body())
        ), 201

    @application.get("/receipts/<receipt_id>")
    def receipt_route(receipt_id):
        receipt = service.receipts.get(receipt_id)

        if receipt is None:
            return jsonify(
                error="Receipt not found"
            ), 404

        return jsonify(receipt), 200

    @application.get("/issuer")
    def issuer_route():
        return jsonify(
            label=service.issuer.label,
            address=service.issuer.address,
            public_key=service.issuer.public_key_hex
        )

    @application.get("/verify")
    def verify_route():
        return jsonify(
            verify_chain=service.verify_chain(),
            invariant_ok=service.invariant_ok(),
            length=len(service.blockchain.chain),
            difficulty=DIFFICULTY
        )

    @application.get("/status")
    def status_route():
        return jsonify(
            service.status()
        )

    @application.get("/chain")
    def full_chain():
        return jsonify(
            chain=service.chain,
            length=len(service.blockchain.chain),
            verified=service.verify_chain()
        )

    @application.get("/transactions/<tx_id>")
    def lookup_route(tx_id):
        result = service.lookup(tx_id)

        if result is None:
            return jsonify(
                error="Transaction not found"
            ), 404

        return jsonify(result), 200

    @application.get("/explorer/rows")
    def explorer_rows_route():
        return jsonify(
            explorer_rows(service.chain)
        )

    @application.get("/explorer")
    def explorer_route():
        return render_template_string(
            EXPLORER_HTML,
            length=len(service.blockchain.chain),
            pending=len(service.pool),
            verified=service.verify_chain(),
            vault_fiat=service.ledger["vault_fiat"],
            circulating=service.ledger["circulating"],
            invariant_ok=service.invariant_ok(),
            blocks=explorer_blocks(service, labels)
        )

    @application.before_request
    def serialise_requests():
        service.lock.acquire()

    @application.teardown_request
    def unlock_requests(error):
        service.lock.release()

    @application.errorhandler(DuplicateTransaction)
    def duplicate_request(error):
        return jsonify(
            error=str(error)
        ), 409

    @application.errorhandler(ValueError)
    @application.errorhandler(KeyError)
    @application.errorhandler(TypeError)
    def invalid_request(error):
        return jsonify(
            error=str(error)
        ), 400

    return application


# Create a fresh payment platform.
platform = RemittancePlatform(reserve_agent)

labels = {
    sender.address: "Sender",
    recipient.address: "Recipient"
}

app = create_app(platform, labels)
client = app.test_client()


def call(method, path, body=None, expected=200):
    response = client.open(
        path,
        method=method,
        json=body
    )

    result = response.get_json()

    print(
        method,
        path,
        "→ HTTP",
        response.status_code
    )

    print(json.dumps(result, indent=2))

    assert response.status_code == expected, result

    return result


print("Fresh payment API ready")
print("All deposits and payouts are simulated")

Fresh payment API ready
All deposits and payouts are simulated


## 6. On-ramp: issue 1,000 mZAR

The sender deposits 1,000 simulated local units.

In [8]:
deposit = call(
    "POST",
    "/onramp",
    {
        "address": sender.address,
        "amount": 1000
    },
    expected=201
)

assert (
    platform.ledger["vault_fiat"]
    == platform.ledger["circulating"]
    == 1000
)

assert platform.invariant_ok()

issuer = call("GET", "/issuer")

mint_record = platform.lookup(
    deposit["tx_id"]
)["transaction"]

assert verify_record(
    mint_record,
    issuer["public_key"]
)

print(
    "Mint signed by the Reserve agent:",
    verify_record(mint_record, issuer["public_key"])
)

print(
    "Sender mZAR:",
    platform.ledger["balances"][sender.address]
)

POST /onramp → HTTP 201
{
  "amount": 1000,
  "block": 1,
  "circulating_after": 1000,
  "deposited_local": 1000,
  "event": "onramp",
  "issuer_signature": "cb6b4f3ee5a89534de2343be7e64f4146d5e76dcb9d85c30d1e9909d75e7da66e6134801c6e5af9231c5a02b32ba25affcf15adb247283660a7d4a065aaacfd9",
  "minted_mzar": 1000,
  "tx_id": "a64acd6c7fa4f9a615bc51ab9dcf02a06f23862aeb28a8b1200a73cfd5ad85e2",
  "vault_fiat_after": 1000
}
GET /issuer → HTTP 200
{
  "address": "954462896f01b2fc449b56fa30968f837d2bb881",
  "label": "Reserve agent (issuer)",
  "public_key": "e2da69abecd42401cab364157212d5a1096903784a1a1a8e03d25eee8510fb6273e11e8a4034691a61d258e643698108bc89e5d3a802cdde3dacc212f3114361"
}
Mint signed by the Reserve agent: True
Sender mZAR: 1000


## 7. Sign and submit 750 mZAR

The sender authorises the payment using their private key. The signed instruction includes a timestamp.

In [9]:
payment = sender.instruction(
    kind="transfer",
    recipient=recipient.address,
    amount=750,
    nonce=platform.ledger["nonces"].get(
        sender.address, 0
    )
)

print(
    "Public-key signature verified:",
    verify_instruction(payment)
)

accepted = call(
    "POST",
    "/transactions/new",
    payment,
    expected=201
)

assert accepted["block_index"] == 2

pending = call(
    "GET",
    "/transactions/" + payment["tx_id"]
)

assert pending["status"] == "pending"

assert (
    platform.ledger["balances"].get(
        recipient.address, 0
    ) == 0
)

print("Recipient confirmed balance before mining: 0")

Public-key signature verified: True
POST /transactions/new → HTTP 201
{
  "block_index": 2,
  "message": "Transaction queued",
  "pending": 1,
  "status": "pending",
  "tx_id": "cc43c3478b3bd6f72db400c0206164c898ab49985f3c0655d67072e05af8cc92"
}
GET /transactions/cc43c3478b3bd6f72db400c0206164c898ab49985f3c0655d67072e05af8cc92 → HTTP 200
{
  "block": null,
  "confirmations": 0,
  "required_confirmations": 2,
  "status": "pending",
  "transaction": {
    "amount": 750,
    "asset": "mZAR",
    "kind": "transfer",
    "nonce": 0,
    "public_key": "0948ae2b0386aae58c64fc0a8685cb0cc4bbef14666f2dc1bfa18c1d016d2ba17aa47d378989f74896168d0f78529d977ba18f46929ae15111957e27764e095d",
    "recipient": "8088396b69b74481b8575fda8a4bd6a176b95f49",
    "sender": "88b0f7e17f7b57b18f4c67eace5e807059200d9e",
    "signature": "9ad0ed0c7a6cf3a3e185e499c22dae26f81ee1dd16bf44cb7a4f8daec3be0047d307cf2670bd4a843c08b6df8cc33b01dcd59692281ec777f5d4c9c1186a7477",
    "timestamp": 1791734214,
    "tx_id": "cc43c

## 8. Mine and verify the payment

In [10]:
mined = call(
    "POST",
    "/mine"
)

print(
    f"Proof of work: difficulty {mined['mining']['difficulty']}, "
    f"{mined['mining']['attempts']} attempts, "
    f"{mined['mining']['seconds']} s"
)

record = call(
    "GET",
    "/transactions/" + payment["tx_id"]
)

assert record["confirmations"] == 1
assert record["status"] == "mined"

print(
    "Recipient mZAR after mining:",
    platform.ledger["balances"][recipient.address]
)


# The recipient authorises redemption.
redemption = recipient.instruction(
    kind="burn",
    recipient="FIAT_VAULT",
    amount=750,
    nonce=0
)


print("\nAttempt redemption too early:")

call(
    "POST",
    "/offramp",
    redemption,
    expected=400
)

assert platform.ledger["vault_fiat"] == 1000


print("\nMine one confirmation block:")

call(
    "POST",
    "/mine"
)

record = call(
    "GET",
    "/transactions/" + payment["tx_id"]
)

assert (
    record["confirmations"]
    >= REQUIRED_CONFIRMATIONS
)

assert record["status"] == "confirmed"

POST /mine → HTTP 200
{
  "hash": "00f15cc7c64a339bf35b59598cce936b26534aed9fdf364df97df1a98e520899",
  "index": 2,
  "message": "New block forged",
  "mining": {
    "attempts": 74,
    "difficulty": 2,
    "nonce": 73,
    "seconds": 0.0024
  },
  "transactions": [
    {
      "amount": 750,
      "asset": "mZAR",
      "kind": "transfer",
      "nonce": 0,
      "public_key": "0948ae2b0386aae58c64fc0a8685cb0cc4bbef14666f2dc1bfa18c1d016d2ba17aa47d378989f74896168d0f78529d977ba18f46929ae15111957e27764e095d",
      "recipient": "8088396b69b74481b8575fda8a4bd6a176b95f49",
      "sender": "88b0f7e17f7b57b18f4c67eace5e807059200d9e",
      "signature": "9ad0ed0c7a6cf3a3e185e499c22dae26f81ee1dd16bf44cb7a4f8daec3be0047d307cf2670bd4a843c08b6df8cc33b01dcd59692281ec777f5d4c9c1186a7477",
      "timestamp": 1791734214,
      "tx_id": "cc43c3478b3bd6f72db400c0206164c898ab49985f3c0655d67072e05af8cc92"
    }
  ]
}
Proof of work: difficulty 2, 74 attempts, 0.0024 s
GET /transactions/cc43c3478b3bd6f72d

## 9. Off-ramp: redeem 750 mZAR

The recipient redeems 750 mZAR for 750 simulated local units.

In [11]:
receipt = call(
    "POST",
    "/offramp",
    redemption,
    expected=201
)

assert receipt["paid_local"] == 750

assert (
    platform.ledger["balances"][recipient.address]
    == 0
)

fetched = call(
    "GET",
    "/receipts/" + receipt["receipt_id"]
)

assert fetched == receipt
assert payment["tx_id"] in receipt["funding_tx_ids"]
assert verify_record(fetched, issuer["public_key"])

print(
    "Recipient simulated payout:",
    receipt["paid_local"],
    "local units"
)

print(
    "Receipt signed by the Reserve agent:",
    verify_record(fetched, issuer["public_key"])
)

print(
    "Receipt funded by payment:",
    payment["tx_id"]
)

POST /offramp → HTTP 201
{
  "address": "8088396b69b74481b8575fda8a4bd6a176b95f49",
  "block": 4,
  "block_hash": "00be0fbc5313a892e48b98897abc812ed595196637a5cfebd160de407f9abb9c",
  "burn_tx_id": "6076ff1c4ba00318c08e94404d0f6a170ea04334c0f0014d43a8714a04482e1c",
  "burned_mzar": 750,
  "circulating_after": 250,
  "currency": "ZAR demo units",
  "funding_tx_ids": [
    "cc43c3478b3bd6f72db400c0206164c898ab49985f3c0655d67072e05af8cc92"
  ],
  "issuer": "954462896f01b2fc449b56fa30968f837d2bb881",
  "paid_local": 750,
  "payout": "SIMULATED",
  "receipt_id": "7176eb03f50f0a50ed55780b79070e35e4feeb11955cf28f0f405a73af749e24",
  "signature": "9c301e283d37692f16618a1089e1a69e851c0460004c4b1e135a9d767d7e93c095773039b315cea37437c6472b9fd28ed6994530fa20a97f382638683c504997",
  "timestamp": 1791734490.3046896,
  "vault_fiat_after": 250
}
GET /receipts/7176eb03f50f0a50ed55780b79070e35e4feeb11955cf28f0f405a73af749e24 → HTTP 200
{
  "address": "8088396b69b74481b8575fda8a4bd6a176b95f49",
  "block"

## 10. Final balances and rejected payments

In [13]:
final_status = call(
    "GET",
    "/status"
)

assert (
    final_status["balances"][sender.address]
    == 250
)

assert (
    final_status["balances"][recipient.address]
    == 0
)

assert (
    final_status["vault_fiat"]
    == final_status["circulating"]
    == 250
)

assert final_status["invariant_ok"]
assert final_status["chain_ok"]


print("\nInsufficient funds: attempt to send 300 mZAR")

overspend = sender.instruction(
    kind="transfer",
    recipient=recipient.address,
    amount=300,
    nonce=1
)

call(
    "POST",
    "/transactions/new",
    overspend,
    expected=400
)


print("\nTampered signature: change a signed amount")

tampered = sender.instruction(
    kind="transfer",
    recipient=recipient.address,
    amount=10,
    nonce=1
)

tampered["amount"] = 20

call(
    "POST",
    "/transactions/new",
    tampered,
    expected=400
)


print("\nReplay: resubmit the mined payment (duplicate → HTTP 409)")

call(
    "POST",
    "/transactions/new",
    payment,
    expected=409
)


assert platform.status() == final_status

print(
    "\nAll rejected attempts left balances "
    "and reserves unchanged."
)

GET /status → HTTP 200
{
  "balances": {
    "8088396b69b74481b8575fda8a4bd6a176b95f49": 0,
    "88b0f7e17f7b57b18f4c67eace5e807059200d9e": 250
  },
  "blocks": 5,
  "chain_ok": true,
  "circulating": 250,
  "invariant_ok": true,
  "nonces": {
    "8088396b69b74481b8575fda8a4bd6a176b95f49": 1,
    "88b0f7e17f7b57b18f4c67eace5e807059200d9e": 1
  },
  "pending": 0,
  "vault_fiat": 250
}

Insufficient funds: attempt to send 300 mZAR
POST /transactions/new → HTTP 400
{
  "error": "Insufficient balance"
}

Tampered signature: change a signed amount
POST /transactions/new → HTTP 400
{
  "error": "Invalid signature or transaction ID"
}

Replay: resubmit the mined payment (duplicate → HTTP 409)
POST /transactions/new → HTTP 409
{
  "error": "Duplicate transaction: already pending or on chain"
}

All rejected attempts left balances and reserves unchanged.


## 11. Explorer and final verification

In [14]:
verification = call(
    "GET",
    "/verify"
)

assert verification["verify_chain"]


# Explorer-style summary for the 750 mZAR transfer (Lecture 10).
summary = platform.lookup(payment["tx_id"])
tx = summary["transaction"]

print("\n=== Payment summary ===")
print("Transaction ID :", summary["tx_id"])
print("Block index    :", summary["block"])
print("Block hash     :", summary["block_hash"])
print("Confirmations  :", summary["confirmations"], f"(required {REQUIRED_CONFIRMATIONS})")
print("From           :", labels[tx["sender"]], tx["sender"])
print("To             :", labels[tx["recipient"]], tx["recipient"])
print("Amount         :", tx["amount"], "mZAR")
print("Status         :", summary["status"])
print("Receipt ID     :", receipt["receipt_id"])
print()


response = client.get("/explorer")

assert response.status_code == 200

display(
    HTML(response.get_data(as_text=True))
)

print("verify_chain():", platform.verify_chain())
print("invariant_ok():", platform.invariant_ok())


# Alter a copy of the recorded payment.
changed_chain = copy.deepcopy(platform.chain)

changed_chain[2]["transactions"][0]["amount"] = 751

assert not platform.verify_chain(changed_chain)

print("Altered-chain test: rejected")
print("LIVE DEMONSTRATION COMPLETE")

GET /verify → HTTP 200
{
  "difficulty": 2,
  "invariant_ok": true,
  "length": 5,
  "verify_chain": true
}

=== Payment summary ===
Transaction ID : cc43c3478b3bd6f72db400c0206164c898ab49985f3c0655d67072e05af8cc92
Block index    : 2
Block hash     : 00f15cc7c64a339bf35b59598cce936b26534aed9fdf364df97df1a98e520899
Confirmations  : 3 (required 2)
From           : Sender 88b0f7e17f7b57b18f4c67eace5e807059200d9e
To             : Recipient 8088396b69b74481b8575fda8a4bd6a176b95f49
Amount         : 750 mZAR
Status         : confirmed
Receipt ID     : 7176eb03f50f0a50ed55780b79070e35e4feeb11955cf28f0f405a73af749e24



verify_chain(): True
invariant_ok(): True
Altered-chain test: rejected
LIVE DEMONSTRATION COMPLETE


## 12. Additional checks

In [15]:
test_service = RemittancePlatform()

test_sender = Wallet.create("Test sender")
test_recipient = Wallet.create("Test recipient")

test_app = create_app(test_service)
test_client = test_app.test_client()


# 1. Invalid deposit amounts must be rejected.
for bad_amount in (0, -1, 1.5, True, "100"):
    before = test_service.status()

    result = test_client.post(
        "/onramp",
        json={
            "address": test_sender.address,
            "amount": bad_amount
        }
    )

    assert result.status_code == 400
    assert test_service.status() == before


# 2. Pending payments must reserve available funds.
test_service.onramp(
    test_sender.address,
    100
)

test_service.submit(
    test_sender.instruction(
        "transfer",
        test_recipient.address,
        70,
        0
    )
)

try:
    test_service.submit(
        test_sender.instruction(
            "transfer",
            test_recipient.address,
            40,
            1
        )
    )

    raise AssertionError(
        "Pending overspending was accepted"
    )

except ValueError:
    pass

assert len(test_service.pool) == 1

test_service.mine_pending()

assert test_service.verify_chain()


# 3. A signature cannot authorise another wallet's address.
forged = test_recipient.instruction(
    "transfer",
    test_sender.address,
    1,
    0
)

forged["sender"] = test_sender.address

assert not verify_instruction(forged)


# 4. A rehashed block without valid PoW must be rejected.
invalid_pow = copy.deepcopy(test_service.chain)

invalid_pow[-1]["nonce"] += 1

while test_service.block_hash(
    invalid_pow[-1]
).startswith("0" * DIFFICULTY):
    invalid_pow[-1]["nonce"] += 1

invalid_pow[-1]["hash"] = test_service.block_hash(
    invalid_pow[-1]
)

assert not test_service.verify_chain(invalid_pow)


# 5. An empty chain is invalid.
assert not test_service.verify_chain([])


# 6. Uppercase copies of an address are rejected (they would be unspendable).
upper = test_sender.instruction(
    "transfer",
    test_recipient.address.upper(),
    1,
    test_service.ledger["nonces"].get(test_sender.address, 0)
)

assert test_client.post(
    "/transactions/new",
    json=upper
).status_code == 400

assert test_client.post(
    "/onramp",
    json={"address": test_sender.address.upper(), "amount": 5}
).status_code == 400


# 7. Forged mints are rejected.
fake_body = {
    "kind": "mint",
    "asset": ASSET,
    "recipient": test_sender.address,
    "amount": 1_000_000,
    "timestamp": time.time(),
    "event_id": "forged",
    "issuer": test_service.issuer.address
}

attacker = Wallet.create("Attacker")

wrong_key_mint = {
    **fake_body,
    "signature": attacker.sign_record(fake_body),
    "tx_id": digest(fake_body)
}

for fake_mint in (
    {**fake_body, "tx_id": digest(fake_body)},     # unsigned
    wrong_key_mint                                  # signed by the wrong key
):
    try:
        test_service.apply(
            test_service.empty_ledger(),
            fake_mint
        )
        raise AssertionError("Forged mint was accepted")
    except ValueError:
        pass

# Inserted into a copy of the chain and fully re-mined: still rejected.
forged_chain = copy.deepcopy(test_service.chain)
forged_chain[1]["transactions"].append(wrong_key_mint)

for i in range(1, len(forged_chain)):
    forged_chain[i]["previous_hash"] = forged_chain[i - 1]["hash"]
    forged_chain[i]["nonce"] = 0

    while not test_service.block_hash(
        forged_chain[i]
    ).startswith("0" * DIFFICULTY):
        forged_chain[i]["nonce"] += 1

    forged_chain[i]["hash"] = test_service.block_hash(
        forged_chain[i]
    )

assert not test_service.verify_chain(forged_chain)


# 8. Redemption works while another wallet has a pending transfer.
test_service.mine_pending()                    # confirm the 70 mZAR transfer

other = Wallet.create("Other wallet")
test_service.onramp(other.address, 50)

other_tx = other.instruction(
    "transfer",
    test_sender.address,
    20,
    0
)

test_service.submit(other_tx)

assert test_service.lookup(other_tx["tx_id"])["status"] == "pending"

test_receipt = test_service.offramp(
    test_recipient.instruction(
        "burn",
        "FIAT_VAULT",
        70,
        0
    )
)

assert test_receipt["paid_local"] == 70
assert len(test_service.pool) == 1             # other wallet's transfer still pending

test_service.mine_pending()

assert test_service.verify_chain()


# 9. A tampered receipt fails verification.
assert verify_record(
    test_receipt,
    test_service.issuer.public_key_hex
)

assert not verify_record(
    {**test_receipt, "paid_local": 7000},
    test_service.issuer.public_key_hex
)


# 10. append_block rejects a block that does not link to the tip.
bad_block = create_linked_block(
    test_service.blockchain.tip(),
    []
)
bad_block.previous_hash = "f" * 64
bad_block, _, _ = mine_block(bad_block, DIFFICULTY)

try:
    test_service.blockchain.append_block(
        bad_block,
        difficulty=DIFFICULTY
    )
    raise AssertionError("Unlinked block was appended")
except ValueError:
    pass


# 11. The timestamp is signed: changing it breaks the signature.
stamped = test_sender.instruction(
    "transfer",
    test_recipient.address,
    1,
    99
)
stamped["timestamp"] += 60

assert not verify_instruction(stamped)


# 12. Submitting the same pending instruction twice returns HTTP 409.
dup_wallet = Wallet.create("Duplicate test")
test_service.onramp(dup_wallet.address, 10)

dup_tx = dup_wallet.instruction(
    "transfer",
    test_sender.address,
    5,
    0
)

assert test_client.post("/transactions/new", json=dup_tx).status_code == 201
assert test_client.post("/transactions/new", json=dup_tx).status_code == 409


# 13. Hashing is deterministic: the same block contents always give the same hash.
stored_genesis = test_service.blockchain.chain[0]

assert (
    Block.from_dict(stored_genesis.to_dict()).compute_hash()
    == stored_genesis.hash
)


# 14. The original demonstration remains unchanged.
assert platform.status() == final_status

print("Additional checks passed")
print("Presentation state preserved")

Additional checks passed
Presentation state preserved


## 13. Optional localhost explorer

The same application can display the explorer in a browser at http://127.0.0.1:5000/explorer (Lecture 08: bind to 127.0.0.1, never 0.0.0.0).

In [18]:
RUN_LOCAL_SERVER = True

if RUN_LOCAL_SERVER:
    from werkzeug.serving import make_server

    # Stop an earlier server if this cell is rerun.
    if "demo_server" in globals():
        demo_server.shutdown()
        demo_thread.join(timeout=5)
        demo_server.server_close()

    demo_server = make_server(
        "127.0.0.1",
        5000,
        app
    )

    demo_thread = threading.Thread(
        target=demo_server.serve_forever,
        daemon=True
    )

    demo_thread.start()

    # Confirm that the server responds.
    with urlopen(
        "http://127.0.0.1:5000/status",
        timeout=5
    ) as response:
        status = json.load(response)

    assert status["chain_ok"]
    assert status["invariant_ok"]

    print("Local server running.")
    print("Chain verification:", status["chain_ok"])
    print("Reserve invariant:", status["invariant_ok"])
    print("Open this link: http://127.0.0.1:5000/explorer")

else:
    print(
        "Local server skipped. Set RUN_LOCAL_SERVER = True "
        "to open the explorer in a browser."
    )

127.0.0.1 - - [11/Oct/2026 18:14:52] "GET /status HTTP/1.1" 200 -


Local server running.
Chain verification: True
Reserve invariant: True
Open this link: http://127.0.0.1:5000/explorer


In [19]:
# Stop server
if "demo_server" in globals():
    demo_server.shutdown()
    demo_thread.join(timeout=5)
    demo_server.server_close()

    del demo_server, demo_thread

    print("Local server stopped.")

else:
    print("No local server is running.")

Local server stopped.


## Attribution and scope

### Sources and reused components

This prototype builds on the group's semester assignments:

- **A1–A2:** SHA-256 hashing, canonical JSON, block structure, proof of work and tamper detection.
- **A3:** transaction schema (sender, recipient, amount, timestamp, signature) and mempool structure, adapted to hold signed transaction dictionaries.
- **A4:** ECDSA wallet creation, address generation, signing and verification.
- **A5–A6:** `Blockchain`, `append_block`, `verify_chain` and proof-of-work mining.
- **A7:** confirmation-depth approach and discussion of probabilistic finality.
- **A8:** Flask API structure and response fields, HTML explorer and localhost demonstration.

The stablecoin payment flow and reserve-accounting approach (`vault_fiat`, `circulating`, `invariant_ok()`) were guided by
Prof. Jules Mba's BLCH9X2 Lecture 10 slides and live-coding notebook
(6 October 2026).

### Capstone adaptations

The capstone integrates simulated deposits, reserve-backed issuance,
signed transfers, confirmation checks, token redemption and payout receipts.

Additional controls include a Reserve agent that signs every mint and receipt,
nonce validation, duplicate rejection, pending-balance checks, lowercase-address
validation, ledger replay and rejection of invalid instructions.

Design note: Lecture 10 treats on-ramp and off-ramp as issuer-ledger events and
only transfers as chain events. This prototype also records mints and burns on
the chain, so every change to reserves has on-chain, signed evidence.
Redemption takes a signed burn instruction, so only the owner of the funds can redeem.

### AI declaration

OpenAI ChatGPT and Anthropic Claude assisted with code adaptation, integration,
review, debugging and testing. Group members are responsible for reviewing,
understanding and verifying the submitted implementation and explaining it during
the live demonstration. Individual contributions are recorded in the report.

### Scope and limitations

All deposits, reserves and payouts are simulated. The South Africa–Zimbabwe
scenario provides the remittance context; actual foreign-exchange conversion
and real cross-border payouts are outside scope.

The prototype operates in one process and stores its state in memory.
It does not implement distributed consensus, multi-node fork resolution,
real bank custody or AML/KYC controls. The `/onramp` endpoint is open on
localhost and stands in for a trusted issuer; in production it would sit
behind KYC/AML checks and operator authentication.

Chain verification establishes record consistency, not the existence
of real bank reserves. The two-confirmation requirement and mining
difficulty of 2 are classroom settings, not production security guarantees.

No mining reward is issued in mZAR because unbacked issuance would
violate the 1:1 reserve rule.